# AgentRecall → Hindsight: project corrections as a governed memory source

[AgentRecall](https://github.com/Goldentrii/AgentRecall) (MIT, local-first) captures an AI
coding agent's **corrections** — a mistake plus the rule that fixes it — and *governs* them:
which are still active, their severity and authority weight, how often they recur.

This recipe imports those corrections into a **Hindsight** bank so that `recall` / `reflect`
surface the corrected understanding at the start of a *new* session — and the agent stops
repeating the mistake.

**Honest division of labor** (this recipe does not blur it):

| | owns |
|---|---|
| **AgentRecall** | correction **capture + governance** — `active` / `weight` / `recurrence` / retraction |
| **Hindsight** | belief synthesis + cross-session **recall** — the engine |

We do **not** claim AgentRecall adds correction-learning: Hindsight's Observations layer
already consolidates corroborating facts natively. AgentRecall decides *what is worth
remembering and still true*; Hindsight decides *what it means and surfaces it later*.

## Before you start

```bash
pip install -r requirements.txt
```

You also need a running Hindsight. The quickest path is the local server (see the
[Hindsight quickstart](https://github.com/vectorize-io/hindsight)); it defaults to
`http://localhost:8888`.

This notebook is **safe to read without a server** — every cell that calls Hindsight is
guarded by a `LIVE` flag. Flip it on only once your server is up:

```bash
HINDSIGHT_LIVE=1 jupyter lab     # or set os.environ["HINDSIGHT_LIVE"] = "1" below
```

> **Security note.** Corrections can quote secrets or PII (a hardcoded key, an internal
> hostname). They are a *net-new egress path*: they do **not** flow through AgentRecall's
> normal sync scrub (which is itself fail-*open*). So this recipe runs a **fail-closed**
> secret scrub before every `retain()`, defaults to **localhost** (nothing leaves your
> machine), and requires an explicit `AR_HINDSIGHT_CLOUD=1` opt-in before any cloud egress.

In [ ]:
import json, os
import import_corrections as ar  # the helper module shipped with this recipe

# Flip to "1" (or `export HINDSIGHT_LIVE=1`) once a Hindsight server is reachable.
LIVE = os.environ.get("HINDSIGHT_LIVE") == "1"
print("LIVE =", LIVE, "| base_url =",
      os.environ.get("HINDSIGHT_BASE_URL", "http://localhost:8888"))

## Step 1 — What an AgentRecall correction looks like

Each correction is one JSON record. The fields that make it *governable* are `active`
(was it retracted?), `severity` (`p0`/`p1`), `weight` (authority), and `recurrence_count`
(how often the agent re-made this mistake). We use the bundled fixture so this runs with
**zero AgentRecall install**.

In [ ]:
records = ar.load_corrections(sample=True)
print(f"{len(records)} corrections in the fixture\n")
print(json.dumps(records[0], indent=2, ensure_ascii=False))

## Step 2 — Quality gate

Real correction corpora are noisy. On a live store **~74% of records are retracted**
(`active: false`), and a few have empty or path-only `rule` fields. Importing those would
teach the bank stale or junk beliefs, so we drop them **before** anything is retained.

In [ ]:
kept, dropped = [], []
for r in records:
    (kept if ar.is_durable(r) else dropped).append(r)

print("KEPT (durable, active rules):")
for r in kept:
    print(f"  \u2713 {r['id']}")
print("\nDROPPED by the gate:")
for r in dropped:
    reason = "retracted" if not r.get("active", True) else "empty/path-only rule"
    print(f"  \u2717 {r['id']}  ({reason})")

## Step 3 — Fail-closed secret scrub

AgentRecall's own scrub runs only inside its Supabase sync chokepoint, and it is
*fail-open* (on error it returns the original text). Pushing corrections to Hindsight
bypasses that path entirely — so we re-apply a scrub here that **re-scans its own output
and raises** if any known secret shape survives. Better to abort the import than ship a
token.

The fixture's second correction quotes a (fake) AWS key in its `context`. Watch it get
redacted.

In [ ]:
leaky = next((r for r in kept if "AKIA" in (r.get("context") or "")), None)
if leaky is None:
    print("(no AKIA-bearing record in the kept set — scrub demo skipped)")
else:
    print("BEFORE:", leaky["context"])
    print("AFTER :", ar.scrub_for_cloud(leaky["context"]))
    # The fail-closed guarantee: scrubbed output contains no surviving secret shape.
    assert "AKIA" not in ar.scrub_for_cloud(leaky["context"])
    print("\n\u2713 scrub is fail-closed \u2014 no secret survives into retain()")

## Step 4 — Retain corrections into a per-project bank

Each correction maps to one `retain()` call:

- **`content`** = the corrected behavior. Hindsight *extracts the fact* from it — the raw
  text is **never stored verbatim**.
- **`document_id`** = the correction's `id` field (**not** the filename stem — on a live
  store 69/87 ids diverge from their stem). Re-importing the same id **upserts**: Hindsight
  deletes the old document and its memories, then reprocesses. That makes re-runs idempotent
  and lets a superseded correction cleanly replace the prior version.
- **`bank_id`** = `agentrecall-<project>` for hard per-project isolation. Tags are query
  filters, not a security boundary.
- **`metadata`** carries `severity` / `weight` / `recurrence_count` as strings, plus
  `confidence_basis="authority-weight"` — a label that tells any downstream consumer the
  weight is correction *authority*, not retrieval relevance and not a truth probability, so
  it is never mistaken for a belief score.

`to_retain_kwargs` runs the fail-closed scrub **as it builds each call** — so a correction
that quotes a secret raises `SecretLeakError` *here*, before any network call. If that
happens, redact or retract the offending correction in AgentRecall and re-run.

In [ ]:
plans = [ar.to_retain_kwargs(r) for r in kept]  # scrub runs inside; raises on a leak
print(json.dumps(plans[0], indent=2, ensure_ascii=False))

if LIVE:
    client = ar.resolve_client()
    banks = set()
    for p in plans:
        if p["bank_id"] not in banks:
            try:
                client.create_bank(bank_id=p["bank_id"], name=p["bank_id"],
                                   mission="AgentRecall corrections for this project.")
            except Exception:
                pass  # best-effort: create_bank is not guaranteed idempotent across re-runs
            banks.add(p["bank_id"])
        client.retain(**p)
    print(f"\n\u2713 retained {len(plans)} corrections into {sorted(banks)}")
else:
    print("\n(LIVE off \u2014 not retained. Set HINDSIGHT_LIVE=1 with a server running.)")

## Step 5 — A new session: recall surfaces the corrected understanding

This is the payoff. In a fresh session the agent asks Hindsight what it knows before acting.
Hindsight recalls **semantically** against the *extracted facts*, so a natural-language
question surfaces the relevant correction even when it shares no words with the stored rule —
here, "Can I push … on my own?" matches the push/publish approval rule.

`recall` returns `response.results`, each with a `.text` fact.

> Note: Hindsight `recall` returns **no per-result score** — results are ranked, not scored.
> Don't write confidence-gating logic against recall output.

In [ ]:
if LIVE:
    bank = ar.bank_id_for("demo-app")
    resp = client.recall(bank_id=bank, query="Can I push a version bump to main on my own?")
    if not resp.results:
        print("(no results yet \u2014 Hindsight may still be extracting facts; retry in a moment)")
    for r in resp.results:
        print("\u2022", r.text)
else:
    print("(LIVE off) Expected: the recalled fact reminds the agent that pushing/publishing")
    print("needs explicit human approval \u2014 so it asks first instead of repeating the mistake.")

## Step 6 — Reflect for a governed synthesis

`reflect` runs a read-only agentic reasoning loop over the bank. With a `response_schema`
it returns `structured_output` (and `.text` is empty — never print both). Here we ask for a
short go/no-go the agent can act on.

In [ ]:
if LIVE:
    schema = {
        "type": "object",
        "properties": {
            "allowed": {"type": "boolean"},
            "reason": {"type": "string"},
        },
        "required": ["allowed", "reason"],
    }
    out = client.reflect(bank_id=ar.bank_id_for("demo-app"),
                         query="Is the agent allowed to git push without asking?",
                         response_schema=schema, budget="low")
    print(json.dumps(out.structured_output, indent=2))
else:
    print('(LIVE off) Expected structured_output:')
    print(json.dumps({"allowed": False,
                      "reason": "Explicit human approval is required before push/publish/deploy."},
                     indent=2))

## What you just built — and its limits

**The loop:** AgentRecall captured a correction → the gate kept the durable ones → the
fail-closed scrub protected the egress → Hindsight retained the extracted facts → in a new
session `recall`/`reflect` surfaced the corrected understanding. Re-running upserts by
`document_id`, so the import is idempotent.

**Honest limitations:**
- Hindsight extracts facts; it never returns your rule text verbatim. Treat recall as
  *"what the bank believes,"* not a transcript.
- `recall` has no per-result score; reinforcement of repeated corrections is Hindsight's
  native Observations behavior, not something this recipe adds.
- The scrub catches known *token shapes* (AWS / GitHub / OpenAI / Slack / npm / PEM / JWT).
  It does **not** catch free-form PII (emails, hostnames, connection strings). The localhost
  default is your main protection; think before setting `AR_HINDSIGHT_CLOUD=1`.
- **Retraction is a one-way door.** This recipe only skips `active:false` records *on the way
  in*. A correction retracted in AgentRecall *after* it was already retained is **not**
  removed by re-running the import — the gate drops it, so it is never re-sent, and an upsert
  never fires. To purge an already-pushed fact you must delete it explicitly via Hindsight's
  directive API; that step is intentionally left to you (deleting belief is a deliberate act,
  especially for a secret-bearing correction).